# 02.7 - Dynamic Programming (DP)

## What is DP?
**Recursion + remembering answers.** Use it when a problem has:
1. **Overlapping subproblems** - the same smaller problem is solved again and again.
2. **Optimal substructure** - the best answer is built from best answers of smaller problems.

Two styles:
| Top-down (memoization) | Bottom-up (tabulation) |
|---|---|
| Write the recursion, add `@cache` | Fill a table from the smallest subproblem up |
| Easiest to think of | No recursion limit, often less memory |

## The 5-step recipe (use it for every DP problem)
1. **State:** what does `dp[i]` (or `dp[i][j]`) mean? *Write it in words.*
2. **Recurrence:** how is `dp[i]` computed from smaller states?
3. **Base cases.**
4. **Order:** which states must be computed first?
5. **Answer:** which state holds the final answer?

## Where is DP used?
| Real system | DP problem |
|---|---|
| **`git diff`, `diff`, code review tools** | Longest Common Subsequence |
| **Spell checkers, "Did you mean…?", fuzzy search** | Edit distance (Levenshtein) |
| **Bioinformatics** (DNA/protein alignment) | Sequence alignment (Needleman-Wunsch) |
| **Speech recognition, phone keyboards** | Viterbi algorithm |
| **Finance / resource allocation / cloud budgets** | Knapsack |
| **Route planning, robotics** | Shortest paths (Bellman-Ford, Floyd-Warshall are DP) |
| **Text justification in word processors** | Minimise "badness" of lines |
| **Reinforcement learning** | Bellman equations / value iteration |

## 1. From recursion to DP: the evolution of one problem
**Climbing stairs:** 1 or 2 steps at a time - how many ways to reach step n? `ways(n) = ways(n-1) + ways(n-2)`.

In [1]:
from functools import cache

def ways_recursive(n):                     # 1) brute force: O(2^n)
    if n <= 1:
        return 1
    return ways_recursive(n - 1) + ways_recursive(n - 2)

@cache
def ways_memo(n):                          # 2) top-down DP: O(n) time, O(n) space
    if n <= 1:
        return 1
    return ways_memo(n - 1) + ways_memo(n - 2)

def ways_table(n):                         # 3) bottom-up DP: O(n) time, O(n) space
    dp = [1] * (n + 1)
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

def ways_optimized(n):                     # 4) only keep what you need: O(1) space
    prev2, prev1 = 1, 1
    for _ in range(2, n + 1):
        prev2, prev1 = prev1, prev1 + prev2
    return prev1

print([f(10) for f in (ways_recursive, ways_memo, ways_table, ways_optimized)])
print("n=80:", ways_optimized(80))

[89, 89, 89, 89]
n=80: 37889062373143906


## 2. The main DP families
| Family | State looks like | Examples |
|---|---|---|
| **1D linear** | `dp[i]` = answer for prefix ending at i | Climbing Stairs, House Robber, Decode Ways, Word Break |
| **2D grid** | `dp[r][c]` = answer for reaching cell (r, c) | Unique Paths, Minimum Path Sum |
| **Two sequences** | `dp[i][j]` = answer for `a[:i]`, `b[:j]` | LCS, Edit Distance |
| **Knapsack** | `dp[i][capacity]` | 0/1 Knapsack, Partition Equal Subset Sum, Coin Change |
| **Intervals / substrings** | `dp[i][j]` = answer for `s[i..j]` | Longest Palindromic Substring, Burst Balloons |
| **State machine** | `dp[i][state]` | Stock problems with cooldown / fees |

---
## Solved problems

### 1. Min Cost Climbing Stairs - LC #746 (Easy)
`cost[i]` to step on stair i; start at 0 or 1; reach the top (past the last stair) cheaply.

In [2]:
def min_cost_climbing_stairs(cost):
    # dp[i] = min cost to stand on step i (top = len(cost))
    a, b = 0, 0                                  # dp[0], dp[1]
    for i in range(2, len(cost) + 1):
        a, b = b, min(b + cost[i - 1], a + cost[i - 2])
    return b

assert min_cost_climbing_stairs([10, 15, 20]) == 15
assert min_cost_climbing_stairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6
print("Min Cost Climbing ")

Min Cost Climbing 


### 2. Decode Ways - LC #91 (Medium) 'A'=1 … 'Z'=26. How many ways to decode a digit string?

In [3]:
def num_decodings(s):
    # dp[i] = ways to decode s[:i]
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        if s[i - 1] != "0":                       # single digit 1-9
            dp[i] += dp[i - 1]
        if i >= 2 and 10 <= int(s[i - 2:i]) <= 26:   # two digits 10-26
            dp[i] += dp[i - 2]
    return dp[n]

assert num_decodings("12") == 2          # AB, L
assert num_decodings("226") == 3         # BZ, VF, BBF
assert num_decodings("06") == 0
print("Decode Ways ")

Decode Ways 


### 3. Word Break - LC #139 (Medium) Can `s` be split into dictionary words?

In [4]:
def word_break(s, word_dict):
    words = set(word_dict)
    # dp[i] = True if s[:i] can be segmented
    dp = [False] * (len(s) + 1)
    dp[0] = True
    for i in range(1, len(s) + 1):
        for j in range(i):
            if dp[j] and s[j:i] in words:
                dp[i] = True
                break
    return dp[-1]

assert word_break("leetcode", ["leet", "code"])
assert word_break("applepenapple", ["apple", "pen"])
assert not word_break("catsandog", ["cats", "dog", "sand", "and", "cat"])
print("Word Break ")

Word Break 


### 4. Unique Paths - LC #62 (Medium) - 2D grid DP
Robot moves only right or down on an m×n grid. Number of paths from top-left to bottom-right.

In [5]:
def unique_paths(m, n):
    # dp[r][c] = paths to reach (r, c) = from above + from left
    dp = [[1] * n for _ in range(m)]
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[m - 1][n - 1]

assert unique_paths(3, 7) == 28
assert unique_paths(3, 2) == 3
print("Unique Paths ")

Unique Paths 


### 5. Minimum Path Sum - LC #64 (Medium)
Grid of costs; cheapest path from top-left to bottom-right (right/down moves).

In [6]:
def min_path_sum(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            if r == 0 and c == 0:
                dp[r][c] = grid[r][c]
            elif r == 0:
                dp[r][c] = dp[r][c - 1] + grid[r][c]
            elif c == 0:
                dp[r][c] = dp[r - 1][c] + grid[r][c]
            else:
                dp[r][c] = min(dp[r - 1][c], dp[r][c - 1]) + grid[r][c]
    return dp[-1][-1]

assert min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12
print("Min Path Sum ")

Min Path Sum 


### 6. 0/1 Knapsack - the classic Items with weights and values, bag capacity W. Each item taken at most once. Maximise value. (Budget allocation, cargo loading, cloud resource planning.)

**Idea:** `dp[i][c] = max(skip item i, take item i: dp[i-1][c - w] + v)`. Compressed to 1D by iterating capacity downwards.

In [7]:
def knapsack(weights, values, capacity):
    # dp[c] = best value with capacity c using items seen so far
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        for c in range(capacity, w - 1, -1):     # go BACKWARDS so each item is used once
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[capacity]

# Laptop bag: items (weight kg, value)
assert knapsack([1, 3, 4, 5], [1, 4, 5, 7], 7) == 9      # items of weight 3 + 4
assert knapsack([10, 20, 30], [60, 100, 120], 50) == 220
print("0/1 Knapsack ")

0/1 Knapsack 


### 7. Partition Equal Subset Sum - LC #416 (Medium) - knapsack in disguise
Can the array be split into two subsets with equal sums?

In [8]:
def can_partition(nums):
    total = sum(nums)
    if total % 2:
        return False
    target = total // 2
    reachable = [True] + [False] * target     # reachable[s]: some subset sums to s
    for x in nums:
        for s in range(target, x - 1, -1):
            reachable[s] = reachable[s] or reachable[s - x]
    return reachable[target]

assert can_partition([1, 5, 11, 5])
assert not can_partition([1, 2, 3, 5])
print("Partition Equal Subset Sum ")

Partition Equal Subset Sum 


### 8. Edit Distance - LC #72 (Medium) spell-checker algorithm
Minimum insert/delete/replace operations to turn `word1` into `word2`.

In [9]:
def min_distance(word1, word2):
    m, n = len(word1), len(word2)
    # dp[i][j] = edits to turn word1[:i] into word2[:j]
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i                     # delete everything
    for j in range(n + 1):
        dp[0][j] = j                     # insert everything
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j],      # delete
                                   dp[i][j - 1],      # insert
                                   dp[i - 1][j - 1])  # replace
    return dp[m][n]

assert min_distance("horse", "ros") == 3
assert min_distance("intention", "execution") == 5

# Real use: "Did you mean ...?"
dictionary = ["python", "pandas", "numpy", "flask", "pytorch"]
typo = "pyhton"
print(f"Did you mean: {min(dictionary, key=lambda w: min_distance(typo, w))}?")
print("Edit Distance ")

Did you mean: python?
Edit Distance 


### 9. Longest Palindromic Substring - LC #5 (Medium) 

**Idea:** Interval DP (`dp[i][j] = s[i]==s[j] and dp[i+1][j-1]`) works in O(n²) time/space; the simpler **expand-around-center** gives O(n²) time and O(1) space.

In [10]:
def longest_palindrome(s):
    best = ""
    def expand(l, r):
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l -= 1; r += 1
        return s[l + 1:r]
    for i in range(len(s)):
        for candidate in (expand(i, i), expand(i, i + 1)):   # odd and even length
            if len(candidate) > len(best):
                best = candidate
    return best

assert longest_palindrome("babad") in ("bab", "aba")
assert longest_palindrome("cbbd") == "bb"
assert longest_palindrome("a") == "a"
print("Longest Palindromic Substring ")

Longest Palindromic Substring 


### 10. Best Time to Buy and Sell Stock with Cooldown - LC #309 (Medium) - state machine DP
Unlimited transactions, but after selling you must wait one day.

**Idea:** Track the best profit in each state (holding, just sold, resting) day by day.

In [11]:
def max_profit_cooldown(prices):
    hold, sold, rest = float("-inf"), 0, 0      # profit in each state at the end of the day
    for p in prices:
        hold, sold, rest = (max(hold, rest - p),  # keep holding, or buy today (from rest)
                            hold + p,             # sell today
                            max(rest, sold))      # do nothing (cooldown finished)
    return max(sold, rest)

assert max_profit_cooldown([1, 2, 3, 0, 2]) == 3
assert max_profit_cooldown([1]) == 0
print("Stock with Cooldown ")

Stock with Cooldown 


## More practice on LeetCode
1D: Min Cost Climbing Stairs (#746), House Robber II (#213), Maximum Product Subarray (#152), Palindromic Substrings (#647)
2D / sequences: Unique Paths II (#63), Longest Common Subsequence (#1143 - in 03.1), Interleaving String (#97), Distinct Subsequences (#115)
Knapsack: Coin Change II (#518), Target Sum (#494), Last Stone Weight II (#1049)
Hard: Burst Balloons (#312), Regular Expression Matching (#10)

Also in `03-Interview Patterns/03.1`: Climbing Stairs, House Robber, Coin Change, LIS, LCS, Maximum Subarray.

## Key takeaways
- DP = recursion with memory. Start with the recursive brute force, then add `@cache`, then convert to a table if needed.
- Always **write the state in words** first: *"dp[i] = number of ways to decode the first i characters"*.
- Knapsack-type: iterate capacity **backwards** for 0/1 items, **forwards** for unlimited items.

**Next: `02.8-Bit-Manipulation.ipynb`**